# Exploratory Data Analysis: Cars and MSRP

This notebook cleans and explores the Kaggle **Car Features and MSRP** dataset. It includes summary statistics and visualizations to investigate vehicle characteristics and their relationship with MSRP (manufacturer's suggested retail price).

**Dataset:** Download `data.csv` from the Kaggle Car Features and MSRP dataset and place it beside this notebook, or upload it when prompted. The notebook does not assume that a data file is already present in the repository.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)

## 2. Load the dataset

The common Kaggle download is named `data.csv`. If the file is named differently, set `DATA_PATH` to its location.

In [ ]:
DATA_PATH = Path('data.csv')
if not DATA_PATH.exists():
    candidates = list(Path('.').glob('*.csv'))
    if candidates:
        DATA_PATH = candidates[0]
    else:
        try:
            from google.colab import files
            uploaded = files.upload()
            csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
            if not csv_files:
                raise FileNotFoundError('Upload a CSV file containing the cars dataset.')
            DATA_PATH = Path(csv_files[0])
        except ImportError as exc:
            raise FileNotFoundError(
                'Cars dataset CSV not found. Download data.csv from Kaggle and place it beside this notebook.'
            ) from exc

df = pd.read_csv(DATA_PATH)
# Remove whitespace accidentally included in column labels.
df.columns = df.columns.str.strip()
print(f'Loaded {DATA_PATH}: {df.shape[0]:,} rows × {df.shape[1]} columns')
display(df.head())

## 3. Understand the data

The features describe the make/model, model year, engine, transmission, drivetrain, fuel economy, and MSRP. MSRP is the target price examined in this exploratory analysis.

In [ ]:
display(df.info())
display(df.describe(include='all').T)

## 4. Remove irrelevant columns and rename

Following the assignment instructions, remove the specified descriptive columns when they are present. This is a choice for this exercise—not a universal rule: such features may be useful in other analyses or predictive models.

In [ ]:
cols_to_drop = ['Engine Fuel Type', 'Market Category', 'Vehicle Style',
                'Popularity', 'Number of Doors', 'Vehicle Size']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

rename_cols = {
    'Engine HP': 'HP', 'Engine Cylinders': 'Cylinders',
    'Transmission Type': 'Transmission', 'Driven_Wheels': 'Drive Mode',
    'highway MPG': 'Highway MPG', 'city mpg': 'City MPG', 'MSRP': 'Price'
}
df = df.rename(columns=rename_cols)
display(df.head())
print('Columns:', df.columns.tolist())

## 5. Duplicate and missing-value checks

Count and remove duplicate rows. Then inspect missingness and drop rows with missing values for this simple baseline analysis. Dropping rows can introduce bias if missingness is systematic, so consider imputation for a real modeling workflow.

In [ ]:
print('Rows before duplicate removal:', len(df))
print('Duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().copy()
print('Rows after duplicate removal:', len(df))

missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame('Missing values'))
df = df.dropna().copy()
print(f'After dropping missing rows: {df.shape[0]:,} rows × {df.shape[1]} columns')
display(df.describe().T)

## 6. Outlier inspection and IQR filtering

Boxplots help reveal unusually high values. The IQR rule below creates `df2` by retaining rows within the 1.5×IQR bounds for **all numeric columns**. This can be aggressive when applied across many features; `df` remains available as the cleaned, unfiltered dataset for the visual exploration below.

In [ ]:
for col in ['Price', 'HP']:
    if col in df.columns:
        fig, ax = plt.subplots(figsize=(10, 2.5))
        sns.boxplot(x=df[col], ax=ax, color='coral')
        ax.set_title(f'Boxplot of {col}')
        plt.tight_layout()
        plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print('Numeric columns:', numeric_cols)

Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Keep rows whose numeric values are within every column's IQR bounds.
mask = ~((df[numeric_cols] < lower) | (df[numeric_cols] > upper)).any(axis=1)
df2 = df.loc[mask].copy()
print(f'Original cleaned data shape: {df.shape}; after IQR filtering: {df2.shape}')

## 7. Univariate distributions

Histograms show the distributions of numeric variables. MSRP and horsepower are often right-skewed because a smaller number of luxury or high-performance vehicles have very high values.

In [ ]:
plot_cols = [c for c in ['Price', 'HP', 'Cylinders', 'Year', 'Highway MPG', 'City MPG'] if c in df.columns]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, col in zip(axes.flat, plot_cols):
    sns.histplot(data=df, x=col, kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribution of {col}')
for ax in axes.flat[len(plot_cols):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

### Vehicle makes

The bar chart shows the 15 makes with the most records in this dataset. Counts describe the dataset's composition, not the overall market share of manufacturers.

In [ ]:
if 'Make' in df.columns:
    top_makes = df['Make'].value_counts().head(15).sort_values()
    ax = top_makes.plot(kind='barh', figsize=(10, 7), color='teal')
    ax.set(title='15 Most Frequent Car Makes', xlabel='Number of cars', ylabel='Make')
    plt.tight_layout()
    plt.show()

### Transmission and drivetrain

A count plot compares transmission categories, with drivetrain indicated by color (hue).

In [ ]:
if {'Transmission', 'Drive Mode'}.issubset(df.columns):
    plt.figure(figsize=(12, 6))
    sns.countplot(data=df, y='Transmission', hue='Drive Mode',
                  order=df['Transmission'].value_counts().index)
    plt.title('Cars by Transmission and Drive Mode')
    plt.xlabel('Number of cars')
    plt.ylabel('Transmission')
    plt.legend(title='Drive mode', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

## 8. Bivariate relationships

The scatterplot compares horsepower with price. Transparency helps reveal dense regions of overlapping observations.

In [ ]:
if {'HP', 'Price'}.issubset(df.columns):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x='HP', y='Price', alpha=0.35, s=25)
    plt.title('MSRP vs. Horsepower')
    plt.xlabel('Horsepower (HP)')
    plt.ylabel('MSRP (Price)')
    plt.tight_layout()
    plt.show()

### Average price by cylinders

This bar plot shows mean MSRP for each engine-cylinder count. Error bars indicate uncertainty around the estimated mean (using seaborn's default interval). Categories with few observations should be interpreted cautiously.

In [ ]:
if {'Cylinders', 'Price'}.issubset(df.columns):
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df, x='Cylinders', y='Price', color='cornflowerblue', errorbar=('ci', 95))
    plt.title('Mean MSRP by Engine Cylinders')
    plt.xlabel('Number of cylinders')
    plt.ylabel('Mean MSRP')
    plt.tight_layout()
    plt.show()

### Price by transmission

Horizontal boxplots compare the distribution of prices across transmission types. The logarithmic x-axis makes it easier to compare categories when prices span a wide range.

In [ ]:
if {'Transmission', 'Price'}.issubset(df.columns):
    plt.figure(figsize=(11, 6))
    order = df.groupby('Transmission')['Price'].median().sort_values().index
    sns.boxplot(data=df, y='Transmission', x='Price', order=order, color='lightgreen')
    plt.xscale('log')
    plt.title('Price Distribution by Transmission (log scale)')
    plt.xlabel('MSRP (log scale)')
    plt.ylabel('Transmission')
    plt.tight_layout()
    plt.show()

## 9. Correlation heatmap

Pearson correlations summarize linear relationships among numeric features. Correlation does not imply causation, and a weak linear correlation does not rule out a nonlinear relationship.

In [ ]:
corr = df.select_dtypes(include=np.number).corr()
display(corr.round(2))
plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap='BrBG', annot=True, fmt='.2f', center=0,
            linewidths=0.5, square=False)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()

## 10. Findings and limitations

Use the generated figures and summary tables to state findings for the dataset you loaded. Typical patterns to check include:

- Whether MSRP has a right-skewed distribution and a small number of expensive outliers.
- Whether horsepower tends to increase with MSRP.
- Whether mean prices differ across engine-cylinder counts or transmission types.
- Which makes are most represented in this dataset.
- Which numeric variables have the strongest positive or negative correlations with price.

**Limitations:** This dataset's records are not necessarily a random sample of cars or sales. MSRP is not the same as transaction price. The assignment's column removal and complete-case approach may discard useful information, and IQR filtering can remove legitimate luxury/performance vehicles. Correlation alone cannot establish cause and effect.